# POC PyCaret - Prédiction de la volatilité du Bitcoin

Ce notebook évalue PyCaret pour la prédiction de la volatilité du Bitcoin à 7 et 30 jours.

L’expérimentation applique le protocole commun défini précédemment : 
- mêmes échantillons, 
- mêmes variables explicatives, 
- découpage temporel sans fuite, 
- budget d’entraînement identique,
- évaluation avec la MAE et la RMSE.

Pendant le POC, les modèles sont entraînés sur l’ensemble d’entraînement et comparés sur l’ensemble de validation. Le jeu de test reste réservé à l’évaluation finale du modèle retenu.

## 1.Vérifier l’environnement

PyCaret 3.3.2 a été retenu, car il s’agit de la dernière version stable compatible avec Python 3.10 au moment de l’expérimentation. La version a été figée afin d’assurer la reproductibilité du POC.
- source : https://pypi.org/project/pycaret/3.3.2/

In [2]:
import platform
import sys

print("Version Python :", sys.version)
print("Exécutable Python :", sys.executable)
print("Système :", platform.platform())

Version Python : 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
Exécutable Python : d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI\.venv_pycaret\Scripts\python.exe
Système : Windows-10-10.0.26100-SP0


In [3]:
import pycaret

print("Version PyCaret :", pycaret.__version__)

Version PyCaret : 3.3.2


## 2.Charger les données du POC

In [4]:
from pathlib import Path

import pandas as pd

project_root = Path().cwd().parent
sets_dir = (
    project_root 
    / "data" 
    / "exports"
    / "datasets"
    / "benchmark_splits")

print(project_root)
print(sets_dir)

d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI
d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI\data\exports\datasets\benchmark_splits


In [26]:
# charger les données du poc
dataset_names = [
    "train_7d", 
    "validation_7d",
    "test_7d",
    "train_30d",
    "validation_30d",
    "test_30d"
]

datasets = {}

for dataset_name in dataset_names:
    file_path = sets_dir / f"{dataset_name}.csv"
    
    datasets[dataset_name] = pd.read_csv(
        file_path,
        parse_dates=["date"]
    )
    print(
        f"{dataset_name}.csv chargé : ",
        datasets[dataset_name].shape
        
    )

train_7d.csv chargé :  (4228, 17)
validation_7d.csv chargé :  (724, 17)
test_7d.csv chargé :  (590, 17)
train_30d.csv chargé :  (4182, 17)
validation_30d.csv chargé :  (701, 17)
test_30d.csv chargé :  (590, 17)


## 3. Identification des variables et des cibles

Cette étape vérifie les colonnes présentes dans les datasets et identifie la cible propre à chaque horizon.

In [27]:
train_7d = datasets["train_7d"]
validation_7d = datasets["validation_7d"]
test_7d = datasets["test_7d"]

train_30d = datasets["train_30d"]
validation_30d = datasets["validation_30d"]
test_30d = datasets["test_30d"]

print(" Colonnes du dataset à 7 jrs :")
print(train_7d.columns.tolist())

print("\n Colonnes du dataset à 30 jrs :")
print(train_30d.columns.tolist())

columns_only_7d = set(train_7d.columns) - set(train_30d.columns)
columns_only_30d = set(train_30d.columns) - set(train_7d.columns)

print("\n Colonnes cible propre au dataset à 7 jrs :", columns_only_7d)
print("\n Colonnes cible propre au dataset à 30 jrs :", columns_only_30d)

 Colonnes du dataset à 7 jrs :
['date', 'log_return', 'volatility_3d', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'daily_range', 'volume_change', 'market_cap_change', 'rate_mro', 'inflation_rate', 'unemployment_rate', 'monetary_m3_rate', 'log_return_lag_1d', 'log_return_lag_7d', 'log_return_lag_30d', 'target_volatility_7d']

 Colonnes du dataset à 30 jrs :
['date', 'log_return', 'volatility_3d', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'daily_range', 'volume_change', 'market_cap_change', 'rate_mro', 'inflation_rate', 'unemployment_rate', 'monetary_m3_rate', 'log_return_lag_1d', 'log_return_lag_7d', 'log_return_lag_30d', 'target_volatility_30d']

 Colonnes cible propre au dataset à 7 jrs : {'target_volatility_7d'}

 Colonnes cible propre au dataset à 30 jrs : {'target_volatility_30d'}


## 4. Préparer les données pour PyCaret

Nous créons ici les copie des colonne du dataframe sans la colonne "date". Le but étant de ne pas modifier les dataframe d'origine pour descontrôles  chronologiques.

In [28]:
target_7d = "target_volatility_7d"
target_30d = "target_volatility_30d"

features_7d = [
    column 
    for column in train_7d.columns
    if column not in ["date", target_7d]
]

features_30d = [
    column 
    for column in train_30d.columns
    if column not in ["date", target_30d]
]

assert features_7d == features_30d, (
    "Les variables explicatives devraient êtres identiques,"
    "mais elles diffèrent entre les features"
    )

train_7d_model = train_7d.drop(columns="date").copy()
validation_7d_model = validation_7d.drop(columns="date").copy()
test_7d_model = test_7d.drop(columns="date").copy()

train_30d_model = train_30d.drop(columns="date").copy()
validation_30d_model = validation_30d.drop(columns="date").copy()
test_30d_model = test_30d.drop(columns="date").copy()

print(features_7d)
print(features_30d)
print("Nombre de variables explicatives :", len(features_7d))
print("train à 7 jours :", train_7d_model.shape)
print("validation à 7 jours :", validation_7d_model.shape)
print("train à 30 jours :", train_30d_model.shape)
print("validation à 30 jours :", validation_30d_model.shape)
print("test à 7 jours :", test_7d_model.shape)
print("test à 30 jours :", test_30d_model.shape)

['log_return', 'volatility_3d', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'daily_range', 'volume_change', 'market_cap_change', 'rate_mro', 'inflation_rate', 'unemployment_rate', 'monetary_m3_rate', 'log_return_lag_1d', 'log_return_lag_7d', 'log_return_lag_30d']
['log_return', 'volatility_3d', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'daily_range', 'volume_change', 'market_cap_change', 'rate_mro', 'inflation_rate', 'unemployment_rate', 'monetary_m3_rate', 'log_return_lag_1d', 'log_return_lag_7d', 'log_return_lag_30d']
Nombre de variables explicatives : 15
train à 7 jours : (4228, 16)
validation à 7 jours : (724, 16)
train à 30 jours : (4182, 16)
validation à 30 jours : (701, 16)
test à 7 jours : (590, 16)
test à 30 jours : (590, 16)


## 5. Configuration de l’expérience PyCaret à 7 et 30 jours

### A. Configuration de l’expérience PyCaret à 7 jours

L’expérience utilise uniquement l’échantillon d’entraînement pour l’apprentissage et la validation croisée. L’échantillon de validation externe est fourni séparément afin d’évaluer les modèles candidats.

La validation croisée respecte l’ordre temporel et applique une purge de 7 jours entre les sous-ensembles.

In [15]:
from pycaret.regression import RegressionExperiment
from sklearn.model_selection import TimeSeriesSplit

temporal_cv_7d = TimeSeriesSplit(
    n_splits=5,
    gap = 8
)

experiment_7d = RegressionExperiment()

experiment_7d.setup(
    data=train_7d_model,
    test_data=validation_7d_model,
    target=target_7d,
    fold_strategy=temporal_cv_7d,
    session_id=42,
    fold=5,
    data_split_shuffle=False,
    fold_shuffle=False,
    n_jobs=-1,
    verbose=True,
    index=False
)

,Description,Value
0,Session id,42
1,Target,target_volatility_7d
2,Target type,Regression
3,Original data shape,"(4952, 16)"
4,Transformed data shape,"(4952, 16)"
5,Transformed train set shape,"(4228, 16)"
6,Transformed test set shape,"(724, 16)"
7,Numeric features,15
8,Preprocess,True
9,Imputation type,simple


In [16]:
import psutil

print("Version psutil :", psutil.__version__)

Version psutil : 7.2.2


In [18]:
import threading
import time

import psutil

resource_samples_7d = {
    "cpu_percent": [],
    "ram_used_mb": []
}

stop_monitoring_7d = threading.Event()


def monitor_resources_7d():
    psutil.cpu_percent(interval=None)

    while not stop_monitoring_7d.is_set():
        resource_samples_7d["cpu_percent"].append(
            psutil.cpu_percent(interval=None)
        )

        resource_samples_7d["ram_used_mb"].append(
            psutil.virtual_memory().used / (1024 ** 2)
        )

        stop_monitoring_7d.wait(0.2)


monitor_thread_7d = threading.Thread(
    target=monitor_resources_7d,
    daemon=True
)


start_time_7d = time.perf_counter()
monitor_thread_7d.start()


try:
    best_models_7d = experiment_7d.compare_models(
        sort="RMSE",
        n_select=3,
        budget_time=10,
        turbo=True,
        errors="ignore"
    )

    cv_results_7d = experiment_7d.pull().copy()

finally:
    stop_monitoring_7d.set()
    monitor_thread_7d.join()


training_time_7d = time.perf_counter() - start_time_7d
peak_cpu_7d = max(resource_samples_7d["cpu_percent"])
peak_ram_7d = max(resource_samples_7d["ram_used_mb"])


print("Temps d'entraînement :", round(training_time_7d, 2), "secondes")
print("Pic CPU :", round(peak_cpu_7d, 2), "%")
print("Pic RAM utilisée :", round(peak_ram_7d, 2), "Mo")

display(cv_results_7d)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
huber,Huber Regressor,0.0175,0.0007,0.0253,-0.2588,0.0239,0.6151,0.1240
llar,Lasso Least Angle Regression,0.0223,0.0008,0.0270,-0.5228,0.0257,1.3608,0.0260
dummy,Dummy Regressor,0.0223,0.0008,0.0270,-0.5228,0.0257,1.3608,0.0180
lasso,Lasso Regression,0.0223,0.0008,0.0270,-0.5228,0.0257,1.3608,0.0340
en,Elastic Net,0.0223,0.0008,0.0270,-0.5228,0.0257,1.3608,0.0240
lr,Linear Regression,0.0223,0.0010,0.0301,-0.9989,0.0266,0.7790,2.2180
br,Bayesian Ridge,0.0223,0.0010,0.0301,-1.0415,0.0266,0.7787,0.0260
lar,Least Angle Regression,0.0225,0.0010,0.0304,-1.0436,0.0267,0.7849,0.0300
ridge,Ridge Regression,0.0228,0.0010,0.0306,-1.1717,0.0268,0.8026,0.0360
omp,Orthogonal Matching Pursuit,0.0219,0.0010,0.0307,-0.8920,0.0279,0.8018,0.0240


Processing:   0%|          | 0/79 [00:00<?, ?it/s]

Temps d'entraînement : 41.14 secondes
Pic CPU : 100.0 %
Pic RAM utilisée : 15276.83 Mo


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
huber,Huber Regressor,0.0175,0.0007,0.0253,-0.2588,0.0239,0.6151,0.124
llar,Lasso Least Angle Regression,0.0223,0.0008,0.0270,-0.5228,0.0257,1.3608,0.026
dummy,Dummy Regressor,0.0223,0.0008,0.0270,-0.5228,0.0257,1.3608,0.018
lasso,Lasso Regression,0.0223,0.0008,0.0270,-0.5228,0.0257,1.3608,0.034
en,Elastic Net,0.0223,0.0008,0.0270,-0.5228,0.0257,1.3608,0.024
lr,Linear Regression,0.0223,0.0010,0.0301,-0.9989,0.0266,0.7790,2.218
br,Bayesian Ridge,0.0223,0.0010,0.0301,-1.0415,0.0266,0.7787,0.026
lar,Least Angle Regression,0.0225,0.0010,0.0304,-1.0436,0.0267,0.7849,0.030
ridge,Ridge Regression,0.0228,0.0010,0.0306,-1.1717,0.0268,0.8026,0.036
omp,Orthogonal Matching Pursuit,0.0219,0.0010,0.0307,-0.8920,0.0279,0.8018,0.024


**Objectif :**

Faire prédire les trois modèles sur validation_7d, puis comparer leur MAE, leur RMSE et leur temps de prédiction.

In [19]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

validation_results_7d = []


for model in best_models_7d:
    model_name = type(model).__name__

    inference_start = time.perf_counter()

    predictions = experiment_7d.predict_model(
        model,
        data=validation_7d_model,
        verbose=False
    )

    inference_time = time.perf_counter() - inference_start

    y_true = validation_7d_model[target_7d].to_numpy()
    y_pred = predictions["prediction_label"].to_numpy()

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    validation_results_7d.append({
        "model": model_name,
        "mae": mae,
        "rmse": rmse,
        "inference_time_seconds": inference_time,
        "inference_time_per_row_ms": (
            inference_time / len(validation_7d_model)
        ) * 1000
    })


validation_scores_7d = (
    pd.DataFrame(validation_results_7d)
    .sort_values("rmse")
    .reset_index(drop=True)
)

display(validation_scores_7d)

,model,mae,rmse,inference_time_seconds,inference_time_per_row_ms
0,LassoLars,0.015960,0.017888,0.203927,0.281668
1,DummyRegressor,0.015960,0.017888,0.154784,0.213789
2,HuberRegressor,0.016754,0.019753,0.362230,0.500318


**LassoLars** et **DummyRegressor** obtiennent exactement les mêmes résultats. Cela suggère que **LassoLars** produit probablement une prédiction presque constante, comme le modèle de référence **DummyRegressor**.

**Conclusion provisoire** : aucun des modèles PyCaret ne dépasse réellement le modèle naïf sur la validation à 7 jours. C’est un résultat important et honnête pour le benchmark ; nous ne choisissons pas encore le modèle final.

PyCaret ne fournissant pas encore de candidat pertinent à 7 jours, nous devons maintenant tester l’horizon 30 jours, puis comparer PyCaret aux autres solutions AutoML avant toute décision finale.

### B. Configuration de l’expérience PyCaret à 30 jours

L’expérience utilise uniquement l’échantillon d’entraînement pour l’apprentissage et la validation croisée. L’échantillon de validation externe est fourni séparément afin d’évaluer les modèles candidats.

La validation croisée respecte l’ordre temporel et applique une purge de 30 jours entre les sous-ensembles.

In [22]:
from pycaret.regression import RegressionExperiment
from sklearn.model_selection import TimeSeriesSplit

temporal_cv_30d = TimeSeriesSplit(
    n_splits=5,
    gap = 31
)

experiment_30d = RegressionExperiment()

experiment_30d.setup(
    data=train_30d_model,
    test_data=validation_30d_model,
    target=target_30d,
    fold_strategy=temporal_cv_30d,
    session_id=42,
    fold=5,
    data_split_shuffle=False,
    fold_shuffle=False,
    n_jobs=-1,
    verbose=True,
    index=False
)

,Description,Value
0,Session id,42
1,Target,target_volatility_30d
2,Target type,Regression
3,Original data shape,"(4883, 16)"
4,Transformed data shape,"(4883, 16)"
5,Transformed train set shape,"(4182, 16)"
6,Transformed test set shape,"(701, 16)"
7,Numeric features,15
8,Preprocess,True
9,Imputation type,simple


In [23]:
import threading
import time

import psutil

resource_samples_30d = {
    "cpu_percent": [],
    "ram_used_mb": []
}

stop_monitoring_30d = threading.Event()


def monitor_resources_30d():
    psutil.cpu_percent(interval=None)

    while not stop_monitoring_30d.is_set():
        resource_samples_30d["cpu_percent"].append(
            psutil.cpu_percent(interval=None)
        )

        resource_samples_30d["ram_used_mb"].append(
            psutil.virtual_memory().used / (1024 ** 2)
        )

        stop_monitoring_30d.wait(0.2)


monitor_thread_30d = threading.Thread(
    target=monitor_resources_30d,
    daemon=True
)


start_time_30d = time.perf_counter()
monitor_thread_30d.start()


try:
    best_models_30d = experiment_30d.compare_models(
        sort="RMSE",
        n_select=3,
        budget_time=10,
        turbo=True,
        errors="ignore"
    )

    cv_results_30d = experiment_30d.pull().copy()

finally:
    stop_monitoring_30d.set()
    monitor_thread_30d.join()


training_time_30d = time.perf_counter() - start_time_30d
peak_cpu_30d = max(resource_samples_30d["cpu_percent"])
peak_ram_30d = max(resource_samples_30d["ram_used_mb"])


print("Temps d'entraînement :", round(training_time_30d, 2), "secondes")
print("Pic CPU :", round(peak_cpu_30d, 2), "%")
print("Pic RAM utilisée :", round(peak_ram_30d, 2), "Mo")

display(cv_results_30d)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lasso,Lasso Regression,0.0215,0.0007,0.0247,-1.2802,0.0236,0.9593,0.0340
llar,Lasso Least Angle Regression,0.0215,0.0007,0.0247,-1.2802,0.0236,0.9593,0.0180
dummy,Dummy Regressor,0.0215,0.0007,0.0247,-1.2802,0.0236,0.9593,0.0160
en,Elastic Net,0.0215,0.0007,0.0247,-1.2802,0.0236,0.9593,0.0200
et,Extra Trees Regressor,0.0239,0.0011,0.0300,-4.0213,0.0285,0.9702,0.5500
knn,K Neighbors Regressor,0.0263,0.0012,0.0327,-3.4938,0.0310,0.9302,0.0960
omp,Orthogonal Matching Pursuit,0.0263,0.0012,0.0329,-2.9207,0.0275,0.7983,0.0300
huber,Huber Regressor,0.0290,0.0014,0.0336,-3.0898,0.0275,0.8715,0.1760
ridge,Ridge Regression,0.0311,0.0015,0.0362,-5.3850,0.0269,0.9465,0.0480
br,Bayesian Ridge,0.0312,0.0015,0.0364,-5.3720,0.0268,0.9522,0.0220


Processing:   0%|          | 0/79 [00:00<?, ?it/s]

Temps d'entraînement : 36.94 secondes
Pic CPU : 100.0 %
Pic RAM utilisée : 15159.97 Mo


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lasso,Lasso Regression,0.0215,0.0007,0.0247,-1.2802,0.0236,0.9593,0.034
llar,Lasso Least Angle Regression,0.0215,0.0007,0.0247,-1.2802,0.0236,0.9593,0.018
dummy,Dummy Regressor,0.0215,0.0007,0.0247,-1.2802,0.0236,0.9593,0.016
en,Elastic Net,0.0215,0.0007,0.0247,-1.2802,0.0236,0.9593,0.020
et,Extra Trees Regressor,0.0239,0.0011,0.0300,-4.0213,0.0285,0.9702,0.550
knn,K Neighbors Regressor,0.0263,0.0012,0.0327,-3.4938,0.0310,0.9302,0.096
omp,Orthogonal Matching Pursuit,0.0263,0.0012,0.0329,-2.9207,0.0275,0.7983,0.030
huber,Huber Regressor,0.0290,0.0014,0.0336,-3.0898,0.0275,0.8715,0.176
ridge,Ridge Regression,0.0311,0.0015,0.0362,-5.3850,0.0269,0.9465,0.048
br,Bayesian Ridge,0.0312,0.0015,0.0364,-5.3720,0.0268,0.9522,0.022


**Objectif :**

Faire prédire les trois modèles sur validation_30d, puis comparer leur MAE, leur RMSE et leur temps de prédiction.

In [25]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

validation_results_30d = []


for model in best_models_30d:
    model_name = type(model).__name__

    inference_start = time.perf_counter()

    predictions = experiment_30d.predict_model(
        model,
        data=validation_30d_model,
        verbose=False
    )

    inference_time = time.perf_counter() - inference_start

    y_true = validation_30d_model[target_30d].to_numpy()
    y_pred = predictions["prediction_label"].to_numpy()

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    validation_results_30d.append({
        "model": model_name,
        "mae": mae,
        "rmse": rmse,
        "inference_time_seconds": inference_time,
        "inference_time_per_row_ms": (
            inference_time / len(validation_30d_model)
        ) * 1000
    })


validation_scores_30d = (
    pd.DataFrame(validation_results_30d)
    .sort_values("rmse")
    .reset_index(drop=True)
)

display(validation_scores_30d)

,model,mae,rmse,inference_time_seconds,inference_time_per_row_ms
0,Lasso,0.016939,0.018201,0.349604,0.498721
1,LassoLars,0.016939,0.018201,0.186695,0.266327
2,DummyRegressor,0.016939,0.018201,0.252781,0.360601


À 30 jours, les trois modèles donnent exactement les mêmes performances :
- MAE : 0,01693
- RMSE : 0,01820

**Lasso** et **LassoLars** ne font donc pas mieux que **DummyRegressor**. Ils produisent probablement eux aussi une prédiction constante ou presque constante.

**Conclusion** : dans cette configuration, PyCaret ne fournit pas de modèle pertinent, ni à 7 jours ni à 30 jours. Nous conservons ces résultats comme preuve du benchmark, puis nous comparerons avec les autres solutions AutoML.



## 6. Évaluation finale des modèles PyCaret à 7 et 30 jours

### A. Évaluation finale - Horizon 7 jours

À l'issue de l'entraînement, de la validation croisée temporelle et de la sélection des modèles, les trois candidats retenus sont évalués sur la partition de test `test_7d`, préalablement figée dans le protocole commun.

Cette étape vise à mesurer leur capacité de généralisation sur des observations qui n'ont été utilisées ni pour l'entraînement ni pour la sélection des modèles.

L'évaluation repose sur les indicateurs suivants :

- **MAE** : erreur absolue moyenne ;
- **RMSE** : racine de l'erreur quadratique moyenne ;
- **R²** : coefficient de détermination ;
- **Temps d'inférence** : durée nécessaire à la génération des prédictions.

Aucun nouvel entraînement ni ajustement des modèles n'est effectué à cette étape.

### B. Évaluation finale - Horizon 30 jours

La même procédure est appliquée aux trois modèles candidats retenus pour l'horizon 30 jours, à partir de la partition figée `test_30d`.

Les métriques et les temps d'inférence sont calculés selon les mêmes modalités que pour l'horizon 7 jours.

### C. Conservation des résultats

Les résultats des deux horizons sont enregistrés afin d'alimenter le tableau comparatif final des solutions AutoML.

Les mesures de performances prédictives et les indicateurs de ressources sont conservés selon une structure commune à PyCaret et AutoGluon.

Leur interprétation et leur comparaison avec AutoGluon seront réalisées dans un notebook distinct, consacré à l'analyse des performances et à la justification du choix technologique.


### A. Évaluer les trois modèles sur test_7d

In [29]:
# Vérification des modèles sélectionnés à 7 jours
print("Nombre de modèles :", len(best_models_7d))

for i, model in enumerate(best_models_7d, start=1):
    print(f"Modèle {i} :", type(model).__name__)

Nombre de modèles : 3
Modèle 1 : HuberRegressor
Modèle 2 : LassoLars
Modèle 3 : DummyRegressor


In [30]:
# Cible de prédiction
target_7d = "target_volatility_7d"

# Séparation des variables explicatives et de la cible
X_test_7d = test_7d_model.drop(columns=[target_7d])
y_test_7d = test_7d_model[target_7d]

# Vérification
print("X_test_7d :", X_test_7d.shape)
print("y_test_7d :", y_test_7d.shape)
print("Valeurs manquantes :", X_test_7d.isna().sum().sum())

X_test_7d : (590, 15)
y_test_7d : (590,)
Valeurs manquantes : 0


In [31]:
import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Modèles sélectionnés lors de la validation croisée
test_results_7d = []

for model in best_models_7d:

    model_name = type(model).__name__

    start_time = time.perf_counter()

    predictions = experiment_7d.predict_model(
        model,
        data=X_test_7d,
        verbose=False
    )

    inference_time = time.perf_counter() - start_time

    y_pred = predictions["prediction_label"]

    test_results_7d.append({
        "model": model_name,
        "MAE": mean_absolute_error(y_test_7d, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test_7d, y_pred)),
        "R2": r2_score(y_test_7d, y_pred),
        "inference_time_seconds": inference_time
    })

final_results_7d = pd.DataFrame(test_results_7d)

display(final_results_7d)

,model,MAE,RMSE,R2,inference_time_seconds
0,HuberRegressor,0.008293,0.012347,-0.253132,0.444908
1,LassoLars,0.018931,0.020405,-2.422448,0.380458
2,DummyRegressor,0.018931,0.020405,-2.422448,0.156920


### B. Évaluer les trois modèles sur test_30d

In [32]:
# Vérification des modèles sélectionnés à 30 jours
print("Nombre de modèles :", len(best_models_30d))

for i, model in enumerate(best_models_30d, start=1):
    print(f"Modèle {i} :", type(model).__name__)

# Vérification de la partition de test
target_30d = "target_volatility_30d"

X_test_30d = test_30d_model.drop(columns=[target_30d])
y_test_30d = test_30d_model[target_30d]

print("\nX_test_30d :", X_test_30d.shape)
print("y_test_30d :", y_test_30d.shape)
print("Valeurs manquantes :", X_test_30d.isna().sum().sum())

Nombre de modèles : 3
Modèle 1 : Lasso
Modèle 2 : LassoLars
Modèle 3 : DummyRegressor

X_test_30d : (590, 15)
y_test_30d : (590,)
Valeurs manquantes : 0


In [33]:
test_results_30d = []

for model in best_models_30d:

    model_name = type(model).__name__

    start_time = time.perf_counter()

    predictions = experiment_30d.predict_model(
        model,
        data=X_test_30d,
        verbose=False
    )

    inference_time = time.perf_counter() - start_time

    y_pred = predictions["prediction_label"]

    test_results_30d.append({
        "model": model_name,
        "MAE": mean_absolute_error(y_test_30d, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test_30d, y_pred)),
        "R2": r2_score(y_test_30d, y_pred),
        "inference_time_seconds": inference_time
    })

final_results_30d = pd.DataFrame(test_results_30d)

display(final_results_30d)

,model,MAE,RMSE,R2,inference_time_seconds
0,Lasso,0.019914,0.021169,-7.029626,0.386425
1,LassoLars,0.019914,0.021169,-7.029626,0.463464
2,DummyRegressor,0.019914,0.021169,-7.029626,0.179289


### C. Exporter les résultats

In [35]:
print(f"7 jours : {training_time_7d}, {peak_cpu_7d}, {peak_ram_7d}")
print(f"30 jours : {training_time_30d}, {peak_cpu_30d}, {peak_ram_30d}")

7 jours : 41.1370506999956, 100.0, 15276.828125
30 jours : 36.935278999997536, 100.0, 15159.96875


In [36]:
from pathlib import Path
import pandas as pd

# Répertoire des résultats du benchmark
export_dir = project_root / "docs" / "benchmark" / "results_benchmark"
export_dir.mkdir(parents=True, exist_ok=True)

# Ajouter les informations nécessaires à la comparaison
export_7d = final_results_7d.copy()
export_7d.insert(0, "tool", "PyCaret")
export_7d.insert(1, "horizon", "7d")

export_30d = final_results_30d.copy()
export_30d.insert(0, "tool", "PyCaret")
export_30d.insert(1, "horizon", "30d")

# Regrouper les résultats des deux horizons
pycaret_results = pd.concat(
    [export_7d, export_30d],
    ignore_index=True
)

# Mesures globales des ressources pendant l'entraînement
resources_by_horizon = {
    "7d": {
        "training_time_seconds": round(training_time_7d, 2),
        "peak_cpu_percent": round(peak_cpu_7d, 2),
        "peak_ram_used_mb": round(peak_ram_7d, 2)
    },
    "30d": {
        "training_time_seconds": round(training_time_30d, 2),
        "peak_cpu_percent": round(peak_cpu_30d, 2),
        "peak_ram_used_mb": round(peak_ram_30d, 2)
    }
}

# Ajouter les mesures de ressources au tableau
for metric in [
    "training_time_seconds",
    "peak_cpu_percent",
    "peak_ram_used_mb"
]:
    pycaret_results[metric] = pycaret_results["horizon"].map(
        lambda h: resources_by_horizon[h][metric]
    )

# Export CSV
export_path = export_dir / "pycaret_test_results.csv"

pycaret_results.to_csv(
    export_path,
    index=False
)

print("Export terminé :", export_path)
print("Dimensions :", pycaret_results.shape)

display(pycaret_results)

Export terminé : d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI\docs\benchmark\results_benchmark\pycaret_test_results.csv
Dimensions : (6, 10)


,tool,horizon,model,MAE,RMSE,R2,inference_time_seconds,training_time_seconds,peak_cpu_percent,peak_ram_used_mb
0,PyCaret,7d,HuberRegressor,0.008293,0.012347,-0.253132,0.444908,41.14,100.0,15276.83
1,PyCaret,7d,LassoLars,0.018931,0.020405,-2.422448,0.380458,41.14,100.0,15276.83
2,PyCaret,7d,DummyRegressor,0.018931,0.020405,-2.422448,0.156920,41.14,100.0,15276.83
3,PyCaret,30d,Lasso,0.019914,0.021169,-7.029626,0.386425,36.94,100.0,15159.97
4,PyCaret,30d,LassoLars,0.019914,0.021169,-7.029626,0.463464,36.94,100.0,15159.97
5,PyCaret,30d,DummyRegressor,0.019914,0.021169,-7.029626,0.179289,36.94,100.0,15159.97
